# 🧠 Modelo CNN - Clasificación de Piezas Industriales (OPTIMIZADO)
## Transfer Learning con VGG16

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Objetivo:** Entrenar un modelo CNN usando Transfer Learning con VGG16

**⚡ OPTIMIZADO para:**
- GPU A100/T4
- Almacenamiento local (evita lentitud de Drive)
- Entrenamiento rápido: 15-30 minutos

---

## ⚙️ Configuración Inicial y Verificación de GPU

In [ ]:
# Verificar entorno y GPU
import sys
import tensorflow as tf

IN_COLAB = 'google.colab' in sys.modules

print('✅ Ejecutando en Google Colab' if IN_COLAB else '⚠️ No estás en Colab')
print(f'🔥 TensorFlow versión: {tf.__version__}')

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'🚀 GPU disponible: SÍ ✅')
    print(f'💪 GPU: {gpus[0].name}')
    
    # Intentar identificar el tipo de GPU
    try:
        gpu_name = tf.test.gpu_device_name()
        if 'A100' in gpu_name:
            print('🏆 GPU A100 detectada - MÁXIMO RENDIMIENTO')
        elif 'T4' in gpu_name:
            print('⚡ GPU T4 detectada - BUEN RENDIMIENTO')
    except:
        pass
else:
    print('❌ GPU NO disponible - El entrenamiento será LENTO')
    print('⚠️ Ve a: Entorno de ejecución → Cambiar tipo → GPU')

## 📁 Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('\n✅ Google Drive montado correctamente')

## 📚 Importar Librerías

In [ ]:
# Librerías básicas
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime
import shutil
from tqdm import tqdm

# TensorFlow y Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# Métricas y visualización
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# Configuración
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Librerías importadas correctamente')

## 🚀 PASO CRÍTICO: Copiar Datos a Almacenamiento Local

**⚠️ MUY IMPORTANTE:** Google Drive es LENTO para leer muchas imágenes.

Copiaremos los datos al almacenamiento local de Colab para entrenar **10-20x MÁS RÁPIDO**

⏱️ **Tiempo de copia:** 5-8 minutos

💡 **Vale la pena:** Reduce entrenamiento de horas a minutos

In [ ]:
print('📦 COPIANDO DATOS A ALMACENAMIENTO LOCAL')
print('='*60)
print('⏱️ Esto tomará 5-8 minutos')
print('💡 Esto acelerará el entrenamiento de horas a minutos\n')

# Rutas de origen (Google Drive)
DRIVE_BASE = Path('/content/drive/MyDrive/datos')
DRIVE_DATA = DRIVE_BASE / 'DataSet'

# Rutas de destino (Local)
LOCAL_DATA = Path('/content/data')
LOCAL_DATA.mkdir(exist_ok=True)

# Copiar Train Dataset
print('📁 [1/6] Copiando Train Dataset...')
!cp -r {DRIVE_DATA / 'Train_Dataset' / 'images'} /content/data/train_images

print('📁 [2/6] Copiando Train labels...')
!cp {DRIVE_DATA / 'Train_Dataset' / 'labels.csv'} /content/data/train_labels.csv

# Copiar Validation Dataset
print('📁 [3/6] Copiando Validation Dataset...')
!cp -r {DRIVE_DATA / 'Valid_Dataset' / 'images'} /content/data/valid_images

print('📁 [4/6] Copiando Validation labels...')
!cp {DRIVE_DATA / 'Valid_Dataset' / 'labels.csv'} /content/data/valid_labels.csv

# Copiar Test Dataset
print('📁 [5/6] Copiando Test Dataset...')
!cp -r {DRIVE_DATA / 'Test_Dataset' / 'images'} /content/data/test_images

print('📁 [6/6] Copiando Test labels...')
!cp {DRIVE_DATA / 'Test_Dataset' / 'labels.csv'} /content/data/test_labels.csv

print('\n' + '='*60)
print('✅ ¡COPIA COMPLETADA!')
print('='*60)
print(f'📊 Datos locales en: {LOCAL_DATA}')
print('🚀 Ahora el entrenamiento será MUCHO más rápido\n')

## 🔧 Configuración de Parámetros

In [ ]:
# Rutas LOCALES (datos copiados)
LOCAL_DATA = Path('/content/data')
TRAIN_DIR = LOCAL_DATA / 'train_images'
VALID_DIR = LOCAL_DATA / 'valid_images'
TEST_DIR = LOCAL_DATA / 'test_images'

# Carpeta para guardar modelos (en Drive)
MODEL_DIR = Path('/content/drive/MyDrive/datos/modelos')
MODEL_DIR.mkdir(exist_ok=True)

# Hiperparámetros - OPTIMIZADOS PARA GPU
IMG_SIZE = 224  # VGG16 requiere 224x224
BATCH_SIZE = 256  # Aumentado para GPUs potentes (ajusta según tu GPU)
EPOCHS = 15
LEARNING_RATE = 0.0001
NUM_CLASSES = 10

# Verificar rutas
print(f'📁 Directorio de datos LOCAL: {LOCAL_DATA}')
print(f'✅ TRAIN existe: {TRAIN_DIR.exists()}')
print(f'✅ VALID existe: {VALID_DIR.exists()}')
print(f'✅ TEST existe: {TEST_DIR.exists()}')
print(f'\n🎯 Configuración:')
print(f'   • Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}')
print(f'   • Batch size: {BATCH_SIZE} ⚡ (Optimizado)')
print(f'   • Épocas máximas: {EPOCHS}')
print(f'   • Learning rate: {LEARNING_RATE}')
print(f'   • Número de clases: {NUM_CLASSES}')
print(f'\n💾 Modelo se guardará en: {MODEL_DIR}')

## 📊 Preparar Datos con Data Augmentation

In [ ]:
# Data Augmentation para ENTRENAMIENTO
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# SOLO normalización para VALIDACIÓN y TEST
valid_test_datagen = ImageDataGenerator(rescale=1./255)

print('✅ Data Augmentation configurado')
print('\n📊 Transformaciones aplicadas al entrenamiento:')
print('   • Normalización (0-1)')
print('   • Rotación ±20°')
print('   • Desplazamiento 20%')
print('   • Zoom ±20%')
print('   • Flip horizontal')
print('\n📊 Validación/Test:')
print('   • Solo normalización (0-1)')

## 🔄 Crear Generadores de Datos

In [ ]:
print('⏳ Preparando generadores de datos...')

# Leer CSV de labels (ahora desde ubicación local)
train_labels_df = pd.read_csv(LOCAL_DATA / 'train_labels.csv')
valid_labels_df = pd.read_csv(LOCAL_DATA / 'valid_labels.csv')
test_labels_df = pd.read_csv(LOCAL_DATA / 'test_labels.csv')

# Preparar DataFrames
train_df = pd.DataFrame({
    'filename': train_labels_df.iloc[:, 1],
    'category': train_labels_df.iloc[:, 3]
})

valid_df = pd.DataFrame({
    'filename': valid_labels_df.iloc[:, 1],
    'category': valid_labels_df.iloc[:, 3]
})

test_df = pd.DataFrame({
    'filename': test_labels_df.iloc[:, 1],
    'category': test_labels_df.iloc[:, 3]
})

print(f'✅ Train: {len(train_df)} imágenes')
print(f'✅ Valid: {len(valid_df)} imágenes')
print(f'✅ Test: {len(test_df)} imágenes')
print(f'\n🏭 Categorías: {train_df["category"].nunique()}')
print(train_df['category'].unique())

In [ ]:
# Crear generadores
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=TRAIN_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

validation_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=valid_df,
    directory=VALID_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = valid_test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=TEST_DIR,
    x_col='filename',
    y_col='category',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('\n✅ Generadores creados exitosamente')
print(f'\n📊 Batches por época: {len(train_generator)}')
print(f'⏱️ Con datos locales, cada batch será ~0.5-1 segundo ⚡')
print(f'\n📊 Mapeo de clases:')
class_indices = train_generator.class_indices
for class_name, class_idx in class_indices.items():
    print(f'   {class_idx}: {class_name}')

## 🖼️ Visualizar Muestras con Augmentation

In [ ]:
def plot_augmented_images(generator, n_images=8):
    images, labels = next(generator)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    
    class_names = list(generator.class_indices.keys())
    
    for i in range(min(n_images, len(images))):
        axes[i].imshow(images[i])
        label_idx = np.argmax(labels[i])
        axes[i].set_title(f'{class_names[label_idx]}', fontsize=10)
        axes[i].axis('off')
    
    plt.suptitle('Muestras de Entrenamiento con Data Augmentation', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('🎨 Visualizando muestras con augmentation...')
plot_augmented_images(train_generator)

## 🧠 Construir Modelo VGG16

In [ ]:
# Cargar VGG16 preentrenado
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar capas base
base_model.trainable = False

print('✅ VGG16 cargado (pesos de ImageNet)')
print(f'📊 Capas congeladas: {len(base_model.layers)}')

In [ ]:
# Construir modelo completo
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

print('✅ Modelo construido')
print('\n📊 Arquitectura del modelo:')
model.summary()

## ⚙️ Compilar Modelo

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('✅ Modelo compilado')
print(f'\n🎯 Configuración:')
print(f'   • Optimizer: Adam')
print(f'   • Learning rate: {LEARNING_RATE}')
print(f'   • Loss: categorical_crossentropy')
print(f'   • Metrics: accuracy')

## 📊 Configurar Callbacks

In [ ]:
# Nombre único para el modelo
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'vgg16_industrial_classifier_{timestamp}'
model_path = MODEL_DIR / f'{model_name}.keras'

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(model_path),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

print('✅ Callbacks configurados')
print(f'\n💾 Modelo se guardará en:')
print(f'   {model_path}')

## 🚀 ENTRENAR EL MODELO

**⏱️ Tiempo estimado con datos locales + GPU:**
- GPU A100: 10-15 minutos ⚡
- GPU T4: 20-30 minutos ✅

In [ ]:
print('🚀 Iniciando entrenamiento...')
print(f'⏱️ Tiempo estimado: 10-30 minutos (según GPU)')
print('='*60)

# Entrenar
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print('\n' + '='*60)
print('🎉 ¡Entrenamiento completado!')
print('='*60)

## 📈 Visualizar Resultados

In [ ]:
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[0].set_title('Accuracy del Modelo', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[1].set_title('Loss del Modelo', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    best_epoch = np.argmax(history.history['val_accuracy'])
    best_val_acc = history.history['val_accuracy'][best_epoch]
    best_train_acc = history.history['accuracy'][best_epoch]
    
    print('\n🏆 Mejores Resultados:')
    print(f'   • Época: {best_epoch + 1}')
    print(f'   • Train Accuracy: {best_train_acc:.4f} ({best_train_acc*100:.2f}%)')
    print(f'   • Validation Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

plot_training_history(history)

## 🧪 Evaluar en Test Set

In [ ]:
print('🧪 Evaluando modelo en Test Set...')

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f'\n📊 Resultados en Test Set:')
print(f'   • Test Loss: {test_loss:.4f}')
print(f'   • Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 🎯 Matriz de Confusión

In [ ]:
print('🔮 Generando predicciones...')

test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print('✅ Predicciones generadas')

In [ ]:
class_names = list(test_generator.class_indices.keys())

print('📊 Classification Report:\n')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Matriz de Confusión', fontsize=16, fontweight='bold', pad=20)
    plt.colorbar()
    
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45, ha='right')
    plt.yticks(tick_marks, classes)
    
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('Etiqueta Real', fontsize=12)
    plt.xlabel('Etiqueta Predicha', fontsize=12)
    plt.tight_layout()
    plt.show()

print('📊 Matriz de Confusión:')
plot_confusion_matrix(y_true, y_pred, class_names)

## 💾 Guardar Modelo

In [ ]:
# Guardar en formato .h5
model_h5_path = MODEL_DIR / f'{model_name}.h5'
model.save(str(model_h5_path))

print(f'✅ Modelo guardado en:')
print(f'   • {model_path} (Keras)')
print(f'   • {model_h5_path} (H5)')

In [ ]:
# Guardar metadatos
metadata = {
    'model_name': model_name,
    'timestamp': timestamp,
    'architecture': 'VGG16 Transfer Learning',
    'input_shape': [IMG_SIZE, IMG_SIZE, 3],
    'num_classes': NUM_CLASSES,
    'classes': class_names,
    'class_indices': class_indices,
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs_trained': len(history.history['accuracy']),
        'learning_rate': LEARNING_RATE,
        'optimizer': 'Adam'
    },
    'performance': {
        'train_accuracy': float(history.history['accuracy'][-1]),
        'val_accuracy': float(history.history['val_accuracy'][-1]),
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss)
    }
}

metadata_path = MODEL_DIR / f'{model_name}_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'\n✅ Metadatos guardados en:')
print(f'   • {metadata_path}')

## 📊 Resumen Final

In [ ]:
print('='*70)
print('🎉 ENTRENAMIENTO COMPLETADO - RESUMEN FINAL')
print('='*70)
print(f'\n🧠 Modelo: {model_name}')
print(f'\n📊 Performance:')
print(f'   • Train Accuracy: {history.history["accuracy"][-1]:.4f} ({history.history["accuracy"][-1]*100:.2f}%)')
print(f'   • Validation Accuracy: {history.history["val_accuracy"][-1]:.4f} ({history.history["val_accuracy"][-1]*100:.2f}%)')
print(f'   • Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print(f'\n📁 Archivos guardados en Drive:')
print(f'   • {model_name}.keras')
print(f'   • {model_name}.h5')
print(f'   • {model_name}_metadata.json')
print(f'\n🎯 Siguiente paso:')
print(f'   • Notebook 03: Desplegar en AWS SageMaker')
print(f'   • Notebook 04: Crear función Lambda + S3')
print('='*70)